In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import statsmodels.formula.api as smf

# =========================
# H6 FINAL SCRIPT (Colab)
# =========================

# 0) Load data
df = pd.read_excel('/mnt/experiment1_analysis_overall.xlsx')

In [15]:
# ---- Column mapping (IMPORTANT) ----
RESP_COL = 'Resp_ratio_pos'
REPAIR_COL = 'Repair_ratio_pos'

pos_features = [RESP_COL, REPAIR_COL]
neg_features = ['Manip_ratio_neg', 'Avoid_ratio_neg', 'Excuse_ratio_neg',
                'Disrespect_ratio_neg', 'Insinc_ratio_neg']

needed_cols = ['apology_type', 'scenario', 'dFpos', 'dFneg', 'dRes_valence'] + pos_features + neg_features
df = df[needed_cols].dropna()

df['scenario'] = df['scenario'].astype('category')
df['apology_type'] = df['apology_type'].astype('category')

In [16]:
# 1) Helper: bootstrap CI for Pearson r
def bootstrap_r_ci(x, y, n_boot=5000, seed=123):
    rng = np.random.default_rng(seed)
    rs = []
    x = np.asarray(x); y = np.asarray(y)
    n = len(x)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        r, _ = pearsonr(x[idx], y[idx])
        rs.append(r)
    lo = np.percentile(rs, 2.5)
    hi = np.percentile(rs, 97.5)
    return lo, hi

In [17]:
# 2) (A) Correlations: H6 core test
corr_rows = []

In [18]:
# dFpos vs F+ features
for f in pos_features:
    r, p = pearsonr(df['dFpos'], df[f])
    lo, hi = bootstrap_r_ci(df['dFpos'], df[f])
    corr_rows.append(['dFpos', f, r, p, lo, hi])

In [19]:
# dFneg vs F- features
for f in neg_features:
    r, p = pearsonr(df['dFneg'], df[f])
    lo, hi = bootstrap_r_ci(df['dFneg'], df[f])
    corr_rows.append(['dFneg', f, r, p, lo, hi])

corr_df = pd.DataFrame(corr_rows, columns=['Delta', 'Feature', 'r', 'p', 'r_ci_low', 'r_ci_high'])
corr_df.to_csv('/mnt/H6_correlations.csv', index=False)

In [20]:
# 3) (B) OLS regressions with controls (type + scenario)
m1 = smf.ols(f"dFpos ~ {RESP_COL} + {REPAIR_COL} + C(apology_type) + C(scenario)", data=df).fit()
m2 = smf.ols("dFneg ~ Manip_ratio_neg + Avoid_ratio_neg + Excuse_ratio_neg + "
             "Disrespect_ratio_neg + Insinc_ratio_neg + C(apology_type) + C(scenario)", data=df).fit()

with open('/mnt/H6_ols_dFpos_summary.txt', 'w') as f:
    f.write(m1.summary().as_text())
with open('/mnt/H6_ols_dFneg_summary.txt', 'w') as f:
    f.write(m2.summary().as_text())

In [21]:
# 4) (C) Scatter plots with fit line (simple, readable)
def scatter_with_fit(x, y, xlabel, ylabel, title, outpath):
    fit = smf.ols("y ~ x", data=pd.DataFrame({'x': x, 'y': y})).fit()
    xs = np.linspace(np.min(x), np.max(x), 200)
    ys = fit.params['Intercept'] + fit.params['x'] * xs

    fig, ax = plt.subplots(figsize=(5,4))
    ax.scatter(x, y, alpha=0.7)
    ax.plot(xs, ys)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.savefig(outpath, bbox_inches='tight')
    plt.close(fig)

In [22]:
scatter_with_fit(df[RESP_COL], df['dFpos'],
                 RESP_COL, 'dFpos',
                 'H6: dFpos vs Responsibility (fit)',
                 '/mnt/H6_scatter_dFpos_vs_responsibility.png')

scatter_with_fit(df[REPAIR_COL], df['dFpos'],
                 REPAIR_COL, 'dFpos',
                 'H6: dFpos vs Repair (fit)',
                 '/mnt/H6_scatter_dFpos_vs_repair.png')

scatter_with_fit(df['Excuse_ratio_neg'], df['dFneg'],
                 'Excuse_ratio_neg', 'dFneg',
                 'H6: dFneg vs Excuse (fit)',
                 '/mnt/H6_scatter_dFneg_vs_excuse.png')

scatter_with_fit(df['Disrespect_ratio_neg'], df['dFneg'],
                 'Disrespect_ratio_neg', 'dFneg',
                 'H6: dFneg vs Disrespect (fit)',
                 '/mnt/H6_scatter_dFneg_vs_disrespect.png')

print("DONE. Files saved under /mnt/")

DONE. Files saved under /mnt/
